In [15]:
import numpy as np
import pandas as pd

np.random.seed(42)
samples_per_class = 15000   # Total ~60,000 samples
classes = ['BPSK', 'QPSK', '16QAM', '64QAM', '256QAM']

data_list = []

for mod in classes:
    if mod == 'BPSK':
        snr = np.random.uniform(-15, 12, samples_per_class)
        rate = np.random.uniform(0.5, 8, samples_per_class)
        dist = np.random.uniform(1000, 10000, samples_per_class)
        power = np.random.uniform(5, 20, samples_per_class)
        
    elif mod == 'QPSK':
        snr = np.random.uniform(4, 18, samples_per_class) 
        rate = np.random.uniform(1, 15, samples_per_class)
        dist = np.random.uniform(500, 6000, samples_per_class)
        power = np.random.uniform(8, 25, samples_per_class)
        
    elif mod == '16QAM':
        snr = np.random.uniform(10, 24, samples_per_class)
        rate = np.random.uniform(3, 35, samples_per_class)
        dist = np.random.uniform(100, 4000, samples_per_class)
        power = np.random.uniform(12, 28, samples_per_class)
        
    elif mod == '64QAM':
        snr = np.random.uniform(16, 28, samples_per_class)
        rate = np.random.uniform(6, 70, samples_per_class)
        dist = np.random.uniform(50, 2500, samples_per_class)
        power = np.random.uniform(15, 30, samples_per_class)
        
    else:  # 256QAM
        snr = np.random.uniform(22, 35, samples_per_class)
        rate = np.random.uniform(12, 150, samples_per_class)
        dist = np.random.uniform(50, 1200, samples_per_class)
        power = np.random.uniform(18, 30, samples_per_class)
    
    temp = pd.DataFrame({
        'snr_db': snr,
        'distance_m': dist,
        'tx_power_dbm': power,
        'bandwidth_mhz': np.random.uniform(1, 50, samples_per_class),
        'frequency_ghz': np.random.choice([0.9, 2.4, 3.5, 5.8], samples_per_class),
        'target_rate_mbps': rate,
        'velocity_kmh': np.random.uniform(0, 120, samples_per_class),
    })
    
    temp['recommended_modulation'] = mod
    data_list.append(temp)


df = pd.concat(data_list, ignore_index=True)


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Derived features
df['path_loss_db'] = (32.4 + 
                     20 * np.log10(df['frequency_ghz']) + 
                     30 * np.log10(df['distance_m'] / 1000) + 
                     np.random.normal(0, 6, len(df)))

df['effective_snr_db'] = df['tx_power_dbm'] - df['path_loss_db'] - np.random.normal(4, 3, len(df))
df['spectral_efficiency_needed'] = df['target_rate_mbps'] / df['bandwidth_mhz']

df.to_csv('modulation_dataset_balanced.csv', index=False)

print("✅ Highly Balanced Dataset Created!")
print("\nClass Distribution:")
print(df['recommended_modulation'].value_counts())
print("\nPercentage:")
print(round(df['recommended_modulation'].value_counts(normalize=True) * 100, 2))

✅ Highly Balanced Dataset Created!

Class Distribution:
recommended_modulation
QPSK      15000
BPSK      15000
256QAM    15000
16QAM     15000
64QAM     15000
Name: count, dtype: int64

Percentage:
recommended_modulation
QPSK      20.0
BPSK      20.0
256QAM    20.0
16QAM     20.0
64QAM     20.0
Name: proportion, dtype: float64


In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb
import joblib 
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv(r"C:\Users\wwwsh\OneDrive\Desktop\modulation_dataset_balanced.csv")

feature_cols = ['snr_db', 'effective_snr_db', 'distance_m', 'tx_power_dbm',
                'bandwidth_mhz', 'frequency_ghz', 'target_rate_mbps',
                 'velocity_kmh', 'spectral_efficiency_needed']

X = df[feature_cols]
y = df['recommended_modulation']

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = xgb.XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test)) 


joblib.dump(model, 'modulation_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("✅ Model and files saved successfully!")
print("Files created: modulation_model.pkl, scaler.pkl, label_encoder.pkl")

Accuracy: 0.8532
✅ Model and files saved successfully!
Files created: modulation_model.pkl, scaler.pkl, label_encoder.pkl


In [ ]:
import pandas as pd
import joblib
import numpy as np


model = joblib.load('modulation_model.pkl')
scaler = joblib.load('scaler.pkl')
le = joblib.load('label_encoder.pkl')




print("✅ Model loaded successfully!\n")

y
def get_user_input():
    print("\n=== 🎯 Modulation Recommendation System ===\n")
    
    snr = float(input("Enter SNR (dB) [-15 to 35]: "))
    dist = float(input("Enter Distance (meters): "))
    power = float(input("Enter Transmit Power (dBm): "))
    bw = float(input("Enter Bandwidth (MHz): "))
    freq = float(input("Enter Frequency (GHz) [e.g. 2.4]: "))
    rate = float(input("Enter Target Data Rate (Mbps): "))
    vel = float(input("Enter Velocity (km/h) [0 if static]: ") or 0)
    
    return {
        'snr_db': snr,
        'distance_m': dist,
        'tx_power_dbm': power,
        'bandwidth_mhz': bw,
        'frequency_ghz': freq,
        'target_rate_mbps': rate,
        'velocity_kmh': vel
    }


def predict_modulation(user_input):
    df = pd.DataFrame([user_input])
    
    
    df['path_loss_db'] = (32.4 + 
                         20 * np.log10(df['frequency_ghz']) + 
                         30 * np.log10(df['distance_m'] / 1000))
    
    df['effective_snr_db'] = df['tx_power_dbm'] - df['path_loss_db'] - 5
    df['spectral_efficiency_needed'] = df['target_rate_mbps'] / df['bandwidth_mhz']

    feature_cols = ['snr_db', 'effective_snr_db', 'distance_m', 'tx_power_dbm',
                    'bandwidth_mhz', 'frequency_ghz', 'target_rate_mbps',
                    'velocity_kmh', 'spectral_efficiency_needed']
    
    X = df[feature_cols]
    X_scaled = scaler.transform(X)
    
    
    pred = model.predict(X_scaled)
    modulation = le.inverse_transform(pred)[0]
    

    proba = model.predict_proba(X_scaled)
    confidence = np.max(proba) * 100
    
    return modulation, confidence


if __name__ == "__main__":
    while True:
        try:
            user_data = get_user_input()
            mod, conf = predict_modulation(user_data)
            
            print("\n" + "="*65)
            print(f"✅ RECOMMENDED MODULATION : {mod}")
            print(f"📊 Confidence              : {conf:.2f}%")
            print("="*65)
            

            again = input("\nDo you want to test another input? (y/n): ").strip().lower()
            if again != 'y':
                print("Thank you for using the Modulation Recommendation System!")
                break
                
        except ValueError:
            print("❌ Invalid input! Please enter numbers only.")
        except Exception as e:
            print(f"Error: {e}") 

✅ Model loaded successfully!


=== 🎯 Modulation Recommendation System ===



Enter SNR (dB) [-15 to 35]:  28
Enter Distance (meters):  300
Enter Transmit Power (dBm):  22
Enter Bandwidth (MHz):  20
Enter Frequency (GHz) [e.g. 2.4]:  2.4
Enter Target Data Rate (Mbps):  120
Enter Velocity (km/h) [0 if static]:  0



✅ RECOMMENDED MODULATION : 256QAM
📊 Confidence              : 99.99%



Do you want to test another input? (y/n):  y



=== 🎯 Modulation Recommendation System ===



...

Enter SNR (dB) [-15 to 35]:  10
Enter Distance (meters):  1000
Enter Transmit Power (dBm):  30
Enter Bandwidth (MHz):  5000
Enter Frequency (GHz) [e.g. 2.4]:  1.8
Enter Target Data Rate (Mbps):  5
Enter Velocity (km/h) [0 if static]:  0



✅ RECOMMENDED MODULATION : 16QAM
📊 Confidence              : 82.55%



Do you want to test another input? (y/n):  y



=== 🎯 Modulation Recommendation System ===



Enter SNR (dB) [-15 to 35]:  -10
